# **Dataset for Pretrained Transformer**

In [13]:
from datasets import load_dataset
dataset = load_dataset("ag_news")

train_and_validation = dataset["train"].train_test_split(test_size=0.1, seed=2026)

train_set = train_and_validation["train"]
val_set = train_and_validation["test"]
test_set = dataset["test"]

# For missclasification
test_texts = test_set.select(range(len(test_set)))

In [3]:
import numpy as np
import torch
import random

RANDOM_SEED = 2026
np.random.default_rng(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


# **Tokenization and Model**
* Load distilbert-base-uncased
* Convert raw text into input_ids + attention_mask
* Create the DataLoaders

In [4]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
  return tokenizer(
      batch["text"],
      max_length=128,
      padding="max_length",
      truncation=True
  )

test_set = test_set.add_column("idx", list(range(len(test_set))))

train_set = train_set.map(tokenize, batched=True)
val_set = val_set.map(tokenize, batched=True)
test_set = test_set.map(tokenize, batched=True)

train_set.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_set.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_set.set_format(type="torch", columns=["input_ids", "attention_mask", "label", "idx"])

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/108000 [00:00<?, ? examples/s]

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [5]:
# Training loop constants
BATCH_SIZE = 16 # Reduce if out of memory
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
WEIGHT_DECAY = 0.05

In [6]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE)


In [7]:
# Load model
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=4).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters: 66956548


# **Training Loop**
* Using suggested hyperparameters
* Modifications (if any): ...

In [10]:
from torch import nn
from torch.cuda.amp import autocast, GradScaler
from transformers import get_linear_schedule_with_warmup

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

scaler = GradScaler()

total_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = total_steps // 10

scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

for epoch in range(NUM_EPOCHS):
  model.train()
  total_loss = 0

  for batch in train_loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["label"].to(device)

    optimizer.zero_grad()

    # fp16 forward pass
    with autocast(dtype=torch.float16):
      outputs = model(input_ids=input_ids, attention_mask=attention_mask)
      logits = outputs.logits
      loss = criterion(logits, labels)

    # Scaled backward pass
    scaler.scale(loss).backward()

    scaler.step(optimizer)
    scaler.update()

    scheduler.step()

    total_loss += loss.item()

  # Validation
  model.eval()

  total_val_loss = 0
  total_correct = 0
  total_samples = 0

  with torch.no_grad():
    for batch in val_loader:
      input_ids = batch["input_ids"].to(device)
      attention_mask = batch["attention_mask"].to(device)
      labels = batch["label"].to(device)

      with autocast(dtype=torch.float16):
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        loss = criterion(logits, labels)

      total_val_loss += loss.item()

      preds = logits.argmax(dim=1)
      total_correct += (preds == labels).sum().item()
      total_samples += labels.size(0)

  avg_val_loss = total_val_loss / len(val_loader)
  val_accuracy = total_correct / total_samples

  avg_loss = total_loss / len(train_loader)
  print(f"Epoch {epoch+1}: Train Loss = {avg_loss:.4f}, Val Loss = {avg_val_loss:.4f}, Val Accuracy = {val_accuracy:.2f}")


/tmp/ipykernel_844/903842079.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_844/903842079.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_844/903842079.py:55: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1: Train Loss = 0.2751, Val Loss = 0.1665, Val Accuracy = 0.94
Epoch 2: Train Loss = 0.1317, Val Loss = 0.1557, Val Accuracy = 0.95
Epoch 3: Train Loss = 0.0810, Val Loss = 0.1626, Val Accuracy = 0.95


# **Testing the Pretrained Transformer Model**

In [15]:
# Make sure model is on the correct device
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

test_correct = 0

store_misclassified_labels = []

with torch.no_grad():
    for batch in test_loader:
        # Move batch to same device as model
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        preds = logits.argmax(dim=1)

        # Count correct predictions
        test_correct += (preds == labels).sum().item()

        # Find misclassified indices
        wrong_idxs = (preds != labels).nonzero(as_tuple=True)[0]

        # Store misclassified examples
        for i in wrong_idxs:
          global_idx = batch["idx"][i].item()

          store_misclassified_labels.append((
              test_texts["text"][global_idx],
              labels[i].item(),
              preds[i].item()
          ))

# Compute accuracy
test_accuracy = test_correct / len(test_set)
print(f'Test accuracy: {test_accuracy:.4f}')

# Display first 10 misclassified examples
for text, true_label, pred_label in store_misclassified_labels[:10]:
    print("Text:", text)
    print("True label:", true_label)
    print("Predicted label:", pred_label)
    print("-"*50)

Test accuracy: 0.9479
Text: IBM to hire even more new workers By the end of the year, the computing giant plans to have its biggest headcount since 1991.
True label: 3
Predicted label: 2
--------------------------------------------------
Text: Some People Not Eligible to Get in on Google IPO Google has billed its IPO as a way for everyday people to get in on the process, denying Wall Street the usual stranglehold it's had on IPOs. Public bidding, a minimum of just five shares, an open process with 28 underwriters - all this pointed to a new level of public participation. But this isn't the case.
True label: 3
Predicted label: 2
--------------------------------------------------
Text: Venezuela Prepares for Chavez Recall Vote Supporters and rivals warn of possible fraud; government says Chavez's defeat could produce turmoil in world oil market.
True label: 0
Predicted label: 2
--------------------------------------------------
Text: Live: Olympics day four Richard Faulds and Stephen Par